In [1]:
import xarray as xr, numpy as np
from config import *

s = xr.open_zarr(str(PATH_SLIIDERS), chunks=None)
r = s.rho.sel(year=2010).isel(ssp=0, iam=0).load()
print('rho dims:', r.dims, r.shape)
print('rango:', float(r.min()), float(r.max()))

if 'ypc' in s.data_vars:
    y = s.ypc.sel(year=2010).isel(ssp=0, iam=0).load()
    print('ypc dims:', y.dims, y.shape)

rho dims: ('country',) (204,)
rango: 0.0063976311834873 0.6499910186855751
ypc dims: ('seg_ir',) (19714,)


In [2]:
import pandas as pd

y_country = y.groupby(s.seg_country).mean().load()

common = sorted(set(r.country.values) & set(y_country.seg_country.values))
rr = r.sel(country=common).values
yy = y_country.sel(seg_country=common).values

print('paises en comun:', len(common))
print('corr rho vs ypc:      ', np.corrcoef(rr, yy)[0,1].round(3))
print('corr rho vs log(ypc): ', np.corrcoef(rr, np.log(yy))[0,1].round(3))

df = pd.DataFrame({'country': common, 'rho': rr, 'ypc': yy}).sort_values('ypc')
print()
print('mas pobres:'); print(df.head(6).to_string(index=False))
print()
print('mas ricos:'); print(df.tail(6).to_string(index=False))

paises en comun: 199
corr rho vs ypc:       0.97
corr rho vs log(ypc):  0.919

mas pobres:
country      rho        ypc
    COD 0.006398 304.456320
    LBR 0.009391 410.393088
    SOM 0.012477 570.927415
    ERI 0.016670 762.967372
    MOZ 0.018588 897.273454
    MDG 0.019581 910.791872

mas ricos:
country      rho          ypc
    KWT 0.530805 51432.321088
    SGP 0.535508 55177.037115
    NOR 0.559345 57108.594272
    BRN 0.544607 57235.766706
    MAC 0.605125 73342.485210
    QAT 0.649991 81785.366540


In [3]:
print('rho/ypc ratio:', (rr/yy).min().round(8), (rr/yy).max().round(8))
print()
import numpy as np
A = np.vstack([yy, np.ones(len(yy))]).T
m, b = np.linalg.lstsq(A, rr, rcond=None)[0]
print(f'rho ≈ {m:.3e} * ypc + {b:.4f}')
print('R²:', np.corrcoef(rr, m*yy+b)[0,1]**2)

rho/ypc ratio: 7.95e-06 8.883e-05

rho ≈ 9.710e-06 * ypc + 0.0625
R²: 0.9413309678990585


In [4]:
%%bash
grep -rn "rho" ~/inequality-dscim-coastal/pyciam/pyCIAM/*.py | head -20

/home/jovyan/inequality-dscim-coastal/pyciam/pyCIAM/run.py:427:    surge = surge * (1 - inputs.rho)
/home/jovyan/inequality-dscim-coastal/pyciam/pyCIAM/utils.py:268:    out["rho"] = out.ypcc / (out.ypcc + usa_ypcc_ref.sel(year=2000, drop=True))


In [6]:
%%bash
grep -rn "rho" ~/inequality-dscim-coastal/ --include="*.py" --include="*.ipynb" 2>/dev/null | grep -v pyciam/ | head -10

/home/jovyan/inequality-dscim-coastal/inequality/adapt_scenarios.ipynb:13:      "rho dims: ('country',) (204,)\n",
/home/jovyan/inequality-dscim-coastal/inequality/adapt_scenarios.ipynb:24:    "r = s.rho.sel(year=2010).isel(ssp=0, iam=0).load()\n",
/home/jovyan/inequality-dscim-coastal/inequality/adapt_scenarios.ipynb:25:    "print('rho dims:', r.dims, r.shape)\n",
/home/jovyan/inequality-dscim-coastal/inequality/adapt_scenarios.ipynb:44:      "corr rho vs ypc:       0.97\n",
/home/jovyan/inequality-dscim-coastal/inequality/adapt_scenarios.ipynb:45:      "corr rho vs log(ypc):  0.919\n",
/home/jovyan/inequality-dscim-coastal/inequality/adapt_scenarios.ipynb:48:      "country      rho        ypc\n",
/home/jovyan/inequality-dscim-coastal/inequality/adapt_scenarios.ipynb:57:      "country      rho          ypc\n",
/home/jovyan/inequality-dscim-coastal/inequality/adapt_scenarios.ipynb:77:    "print('corr rho vs ypc:      ', np.corrcoef(rr, yy)[0,1].round(3))\n",
/home/jovyan/inequality-dsc

%%bash
pwd && ls

In [12]:
%%bash
pwd && ls

/home/jovyan/inequality-dscim-coastal/inequality
00_finish_run.py
00_finish_v2.py
00_full_run.py
00_resume_run.py
00a_validate_access.ipynb
00b_test_run.py
00b_test_run_and_estimate_costs.ipynb
01_process_slr_inputs.ipynb
02_run_pyciam_inequality.ipynb
02_run_pyciam_inequality_fix.ipynb
03_map_to_gadmid.py
04_subset_coastal_gadmids.py
05_globaladapt.py
README.md
__pycache__
adapt_scenarios.ipynb
config.py
finish_run.log
finish_run_report.json
finish_v2.log
finish_v2_report.json
full_run.log
globaladapt.log
globaladapt_report.json
globaladapt_test.log
resume_run.log
test_run.log
test_run_report.json
tests.ipynb


In [14]:
import xarray as xr
from cloudpathlib import AnyPath

print('PATH_SLR_INEQUALITY =', PATH_SLR_INEQUALITY)
p = AnyPath(str(PATH_SLR_INEQUALITY))
print('existe:', p.exists())

PATH_SLR_INEQUALITY = gs://impactlab-data-scratch/inequality-pyciam/ar6-tlim-slr-1000samples.zarr
existe: False


In [15]:
from cloudpathlib import AnyPath
d = AnyPath('gs://impactlab-data-scratch/inequality-pyciam/')
try:
    for f in sorted(d.iterdir())[:30]:
        print(f.name)
except Exception as e:
    print('vacío o no existe:', e)

In [16]:
!grep -o "gs://[^\"']*" ~/inequality-dscim-coastal/inequality/01_process_slr_inputs.ipynb | sort -u

gs:// to /gcs/ fuse path\n
gs://\
gs://ar6-lsl-simulations-public-standard/gridded/full_sample_workflows/wf_1f/tlim2.0/total-workflow.zarr\
gs://ar6-lsl-simulations-public-standard/gridded/full_sample_workflows/wf_1f/tlim2.0win0.25/total-workflow.zarr\
gs://ar6-lsl-simulations-public-standard/gridded/full_sample_workflows/{workflow}/{tlim}win0.25/total-workflow.zarr`\n
gs://ar6-lsl-simulations-requesterpays-standard/gridded/full_sample_components/verticallandmotion-kopp14-verticallandmotion_localsl.zarr\
gs://ar6-lsl-simulations-requesterpays-standard/gridded/full_sample_components/verticallandmotion-kopp14-verticallandmotion_localsl.zarr`\n
gs://impactlab-data-scratch/inequality-pyciam
gs://impactlab-data-scratch/inequality-pyciam/test-slr.zarr\
gs://impactlab-data/coastal/data/raw/slr/ar6/ar6/global/full_sample_workflows/{workflow}/{tlim}win0.25/total-workflow.nc`\n
gs://impactlab-data/coastal/local-scc-model/data/int/sliiders-ir.zarr\
gs://impactlab-data/coastal/local-scc-model/data

In [17]:
from cloudpathlib import AnyPath
for n in ['pyCIAM_outputs_inequality_1000_ssp234_v2.zarr',
          'pyCIAM_outputs_inequality_1000_ssp234_v2_globaladapt.zarr']:
    p = AnyPath(f'gs://impactlab-data/gcp/outputs/coastal/{n}')
    print(n, '->', p.exists())

s = AnyPath('gs://impactlab-data/coastal/local-scc-model/data/int/sliiders-ir.zarr')
print('sliiders ->', s.exists())

pyCIAM_outputs_inequality_1000_ssp234_v2.zarr -> True
pyCIAM_outputs_inequality_1000_ssp234_v2_globaladapt.zarr -> True
sliiders -> True


In [19]:
%%bash
sed -n '200,290p' ~/inequality-dscim-coastal/pyciam/pyCIAM/io.py


        )

    if not include_cc:
        scen_mc_filter = scen_mc_filter[
            scen_mc_filter.get_level_values("scenario") == ncc_name
        ]
    return scen_mc_filter


def _load_lslr_for_ciam(
    slr_store,
    lonlats,
    interp_years=None,
    scen_mc_filter=None,
    include_ncc=True,
    include_cc=True,
    mc_dim="mc_sample_id",
    lsl_var="lsl_msl05",
    lsl_ncc_var="lsl_ncc_msl05",
    ncc_name="ncc",
    slr_0_year=2005,
    storage_options={},
    quantiles=None,
):
    if scen_mc_filter is None:
        scen_mc_filter = _load_scenario_mc(
            slr_store,
            include_ncc=include_ncc,
            include_cc=include_cc,
            mc_dim=mc_dim,
            storage_options=storage_options,
            quantiles=quantiles,
            ncc_name=ncc_name,
        )

    wcc = scen_mc_filter.get_level_values("scenario") != ncc_name
    scen_mc_ncc = scen_mc_filter[~wcc].droplevel("scenario").values
    scen_mc_xr_wcc = (
        scen_mc_filter[wcc]


In [21]:
%%bash
grep -rn "lsl_var" ~/inequality-dscim-coastal/pyciam/pyCIAM/*.py

/home/jovyan/inequality-dscim-coastal/pyciam/pyCIAM/io.py:217:    lsl_var="lsl_msl05",
/home/jovyan/inequality-dscim-coastal/pyciam/pyCIAM/io.py:255:            slr[lsl_var]


In [ ]:
grep -rn "lsl_var" ~/inequality-dscim-coastal/pyciam/pyCIAM/*.py